# Lesson 03 - Agenttisuunnittelumallit

Tässä oppitunnissa tutkimme kolmea perustavaa suunnittelumallia tehokkaiden tekoälyagenttien rakentamiseen:

1. **Selkeät agentin ohjeet** — Tarkkojen, roolia määrittävien kehotteiden laatiminen, jotka ohjaavat agentin toimintaa
2. **Jäsennelty tuloste Pydantic-malleilla** — Varmistetaan, että agentit palauttavat ennustettavia, validoituja tietoja
3. **Yhden vastuun agentit** — Suunnitellaan kohdennettuja agentteja, jotka kukin tekevät yhden asian hyvin

Sovellamme kutakin mallia **matkakohteiden suositusjärjestelmä** -tilanteeseen, kehittäen järjestelmää vaiheittain, joka voi ehdottaa kohteita, tarkistaa saatavuuden ja hoitaa logistiikan.


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Kuvio 1: Selkeät ohjeet agentille

Vaikuttavin kuvio on myös yksinkertaisin: kirjoittaa selkeät, yksityiskohtaiset ohjeet agentillesi.

Hyvät ohjeet määrittelevät:
- **Kuka** agentti on (persoonallisuus ja sävy)
- **Mitä** sen tulisi tehdä (askel askeleelta vastuut)
- **Miten** sen tulisi käyttäytyä (rajoitukset ja tyyli)

Alla luomme matkaopasagentin selkeillä ohjeilla, jotka muovaavat jokaisen sen tuottaman vastauksen.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Kuvio 2: Rakenteinen tuloste Pydantic-mallien avulla

Vapaa muotoinen teksti on hyödyllistä keskusteluissa, mutta alajärjestelmät tarvitsevat rakenteellista dataa. 
Yhdistämällä **Pydantic-mallit** ja **työkalutoiminto** voimme:

- Määritellä tarkka skeema agentin tulosteelle
- Varmistaa vastausten oikeellisuuden automaattisesti
- Integroimaan agenttien tulokset sovelluslogiikkaan luotettavasti

Esittelemme myös työkalun, joka palauttaa kohdetiedot, jotta agentti voi perustaa suosituksensa todellisiin tietoihin.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Kuvio 3: Yhden Vastuun Agentit

Monimutkaiset tehtävät hyötyvät työn jakamisesta useille keskittyneille agenteille, joilla on yksi vastuualue:

- **Kohdeasiantuntija**, joka tuntee paikat ja saatavuuden
- **Logistiikkasuunnittelija**, joka hoitaa lennot, hotellit ja matkasuunnitelmat

Tämä heijastaa ohjelmistotekniikan periaatetta *vastuiden erottelusta* — jokainen agentti on helpompi testata, ylläpitää ja parantaa itsenäisesti.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Yhteenveto

Tässä oppitunnissa sovelsimme kolmea agenttisuunnittelumallia matkasuositusten skenaarioon:

| Malli | Keskeinen ajatus | Hyöty |
|---|---|---|
| **Selkeät ohjeet** | Määrittele persoona, vastuut ja rajoitukset etukäteen | Johdonmukainen, brändin mukainen agenttikäytös |
| **Rakenneellinen tuloste** | Käytä Pydantic-malleja vastausmuotona | Vahvistetut, koneellisesti luettavat tulokset |
| **Yhden vastuun periaate** | Anna jokaiselle agentille yksi keskittynyt tehtävä | Helpompi testata, ylläpitää ja yhdistellä |

Nämä mallit toimivat luonnollisina kokonaisuuksina — voit yhdistää selkeät ohjeet rakenneellisen tulosteen kanssa yhden vastuun periaatteella varustetussa agentissa rakentaaksesi luotettavia, tuotantovalmiita järjestelmiä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, on hyvä huomioida, että automaattikäännöksissä saattaa esiintyä virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen omalla kielellä tulee pitää ensisijaisena lähteenä. Tärkeissä tiedoissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tästä käännöksestä mahdollisesti aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
